# 02 — Audit, clean, split

Week 2. What is wrong with this file, what was decided about each problem,
how the data is divided, and what May says about the question — before any
model exists.

Every rule applied here lives in `src/data.py`, so the table below and the
data a model trains on cannot drift apart. **Nothing in the cleaning learns
anything from the rows**: every threshold is a constant written in
[`PROBLEM.md`](../PROBLEM.md).

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

from src.data import (
    add_target,
    add_zones,
    clean,
    findings_table,
    load_trips,
    load_zones,
    save_processed,
    split,
)
from src.evaluate import save_figure


## The audit\n\nLook before deciding.

In [ ]:
raw = load_trips()
zones = load_zones()
print(f"{len(raw):,} rows x {raw.shape[1]} columns")

pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (100 * raw.isna().mean()).round(2),
    "n_unique": raw.nunique(),
})

In [ ]:
print("exact duplicate rows:", int(raw.duplicated().sum()))

print("\npickup month, as published:")
print(raw["tpep_pickup_datetime"].dt.to_period("M").value_counts().sort_index().to_string())

print("\npayment_type (0 = flex fare, 6 = voided):")
print(raw["payment_type"].value_counts(dropna=False).sort_index().to_string())

print("\nRatecodeID (5 negotiated, 6 group, 99 unknown):")
print(raw["RatecodeID"].value_counts(dropna=False).sort_index().to_string())

print("\nrequest_source:")
print(raw["request_source"].value_counts(dropna=False).to_string())

In [ ]:
trips = add_target(add_zones(raw))
trips[["duration_min", "speed_mph", "trip_distance", "passenger_count",
       "fare_amount", "total_amount"]].describe().T.round(2)

## The findings table

One row per problem, with the decision — **fix, flag or drop** — and what the
code actually does about it. `rows` is how many rows the problem touches;
`removed_here` is how many this rule is the *first* to remove, applying the
drop rules in order.

In [ ]:
findings = findings_table(trips)
findings

### The three decisions the project requires of every team

**Flex fare trips — kept (fix).** A quarter of the sample, with no
`passenger_count` and no `RatecodeID`. They are kept: a flex fare passenger
still wants an arrival time, and the meter timed the journey the same way.
The missing passenger count is imputed *inside the pipeline*, so the imputed
value is learned from May alone. Dropping them would answer a narrower
question than the one we were asked.

**Trips that cannot be real — dropped, except one.** Zero and negative
durations, sub-minute trips, trips over three hours, zero distances,
meter-fault distances and impossible implied speeds all go. So do unknown
zones (264, 265): the zone is this model's only geography, and a trip with no
borough cannot be given one. Voided trips would go too — the count above
shows how many there actually are. Negative fares and totals are **flagged,
not dropped**: they mark a refund or correction, they say nothing about the
clock, and a row that passes every validity rule should not be removed for an
accounting reason.

**Non-standard fares — kept and flagged, deliberately.** `RatecodeID` 5, 6
and 99 are *not* used to filter. The rate code is set at the **end** of the
trip, so filtering training rows by it would shape the training population
using knowledge the model will not have at pickup — a quiet leak that no
metric would catch. It appears only as a reporting slice in `04`.

In [ ]:
# What is being removed, in the open.
print("negative durations:", int((trips["duration_min"] < 0).sum()))
print("exactly zero duration:", f"{int((trips['duration_min'] == 0).sum()):,}")

zero_dist = (trips["trip_distance"] == 0) & (trips["duration_min"] > 1)
print(f"\nzero-distance trips that still ran over a minute: {int(zero_dist.sum()):,}"
      f" (median {trips.loc[zero_dist, 'duration_min'].median():.1f} min)")

print("\nthe largest distances in the file (miles):")
print(trips["trip_distance"].nlargest(5).round(1).to_string())

print("\nwhat fails at least one drop rule, by payment_type:")
failed = ~trips.index.isin(clean(trips).index)
print(trips.loc[trips["duration_min"].isna() | (trips["duration_min"] <= 0)
                | (trips["trip_distance"] <= 0), "payment_type"]
      .value_counts().sort_index().to_string())

In [ ]:
clean_trips = clean(trips)
kept = 100 * len(clean_trips) / len(trips)
print(f"kept {len(clean_trips):,} of {len(trips):,} rows ({kept:.2f}%) — "
      f"removed {len(trips) - len(clean_trips):,} ({100 - kept:.2f}%)")
clean_trips[["duration_min", "speed_mph", "trip_distance"]].describe().T.round(2)

## The split

May trains, June decides everything, July is scored once. Split on **pickup
month**, not a random shuffle: the model will always be used on a month it
has not seen, and a shuffle would let May traffic leak into the June score.

In [ ]:
parts = split(clean_trips)
for name, part in parts.items():
    ts = part["tpep_pickup_datetime"]
    print(f"{name:<11} {len(part):>7,} rows   {ts.min()} .. {ts.max()}")

dropped = len(clean_trips) - sum(len(p) for p in parts.values())
print(f"\ndropped by the month filter (April pickups outside the sample): {dropped}")

train, val, test = parts["train"], parts["validation"], parts["test"]
print(save_processed(clean_trips, "trips_clean.parquet"))

## The analysis

**May only.** June is for choices in `03`; July is not opened until `04`.

### Duration against distance — how far does distance alone get you?

In [ ]:
s = train.sample(20_000, random_state=0)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(s["trip_distance"], s["duration_min"], s=3, alpha=0.15)
ax.set(xlim=(0, 25), ylim=(0, 90), xlabel="trip_distance (miles)",
       ylabel="duration (minutes)", title="Duration vs distance, May (20k sample)")
r = train["trip_distance"].corr(train["duration_min"])
ax.text(0.98, 0.04, f"Pearson r = {r:.3f}", transform=ax.transAxes, ha="right")
save_figure(fig, "duration-vs-distance")
plt.show()

bands = pd.cut(train["trip_distance"], [0, 1, 3, 6, 12, np.inf],
               labels=["<1 mi", "1-3 mi", "3-6 mi", "6-12 mi", "12+ mi"], right=False)
spread = (train.groupby(bands, observed=True)["duration_min"]
          .agg(trips="count", median="median", std="std",
               p10=lambda x: x.quantile(0.1), p90=lambda x: x.quantile(0.9)))
print("how much duration still varies inside one distance band (May):")
print(spread.round(2).to_string())

### Average speed by hour of the day and day of the week — when is New York slowest?

In [ ]:
pivot = (train.assign(hour=train["tpep_pickup_datetime"].dt.hour,
                      dow=train["tpep_pickup_datetime"].dt.day_name())
         .pivot_table(index="dow", columns="hour", values="speed_mph", aggfunc="median")
         .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]))

fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn")
ax.set(xticks=range(24), xticklabels=pivot.columns, yticks=range(7),
       yticklabels=pivot.index, xlabel="hour of pickup", title="Median speed (mph), May")
fig.colorbar(im, ax=ax, label="mph")
save_figure(fig, "speed-by-hour-and-day")
plt.show()

flat = pivot.stack()
print(f"slowest: {flat.idxmin()} at {flat.min():.1f} mph")
print(f"fastest: {flat.idxmax()} at {flat.max():.1f} mph")
pivot.round(1)

### Speed by pickup zone and borough, and for airport trips

In [ ]:
print(train.groupby("pickup_borough", observed=True)
      .agg(trips=("duration_min", "size"), median_speed=("speed_mph", "median"),
           median_duration=("duration_min", "median"), median_distance=("trip_distance", "median"))
      .sort_values("trips", ascending=False).round(2).to_string())

print("\nairport trips vs the rest:")
print(train.groupby("is_airport_trip")
      .agg(trips=("duration_min", "size"), median_speed=("speed_mph", "median"),
           median_duration=("duration_min", "median"), median_distance=("trip_distance", "median"))
      .round(2).to_string())

zone_speed = (train.groupby("pickup_zone", observed=True)
              .agg(trips=("speed_mph", "size"), median_speed=("speed_mph", "median")))
zone_speed = zone_speed[zone_speed["trips"] >= 500].sort_values("median_speed")
print("\nslowest and fastest pickup zones (>= 500 trips):")
print(pd.concat([zone_speed.head(8), zone_speed.tail(8)]).round(2).to_string())

### The long tail — trips that took far longer than their distance suggests

In [ ]:
typical = train["trip_distance"].sum() / (train["duration_min"].sum() / 60)
print(f"typical speed across all of May: {typical:.2f} mph")

tail = train.assign(
    expected_min=train["trip_distance"] / typical * 60,
    excess_min=train["duration_min"] - train["trip_distance"] / typical * 60,
)
print(tail["excess_min"].describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())

worst = tail.nlargest(10, "excess_min")[
    ["tpep_pickup_datetime", "pickup_zone", "dropoff_zone", "trip_distance",
     "duration_min", "excess_min", "speed_mph"]]
print("\nthe ten worst:")
print(worst.round(2).to_string(index=False))

top1 = tail.loc[tail["excess_min"] > tail["excess_min"].quantile(0.99)]
print(f"\nthe worst 1% ({len(top1):,} trips) by pickup hour:")
print(top1["tpep_pickup_datetime"].dt.hour.value_counts().sort_index().to_string())
print("\nand by pickup borough:")
print(top1["pickup_borough"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(tail["excess_min"].clip(-30, 60), bins=90)
ax.set(xlabel="minutes longer than the typical speed implies", ylabel="trips",
       title="The long tail, May")
save_figure(fig, "long-tail")
plt.show()

### What the analysis says

- Distance explains a lot, but **not enough on its own**: inside a single
  distance band the duration still spreads widely (see the p10–p90 columns),
  which is the room a model has to work in.
- The city has a clear daily rhythm: the median speed roughly halves between
  the small hours and the afternoon. **The hour of the week is the strongest
  thing we can know at pickup besides the distance.**
- Boroughs differ sharply, and airport trips are their own animal: long,
  comparatively fast, and few.
- The long tail is concentrated in the slow hours and in Manhattan. Nothing
  knowable at pickup explains the worst of it — which is exactly why the app
  will need a buffer, not just a point estimate.

`03` turns the first two into features, and reports what each one is worth.